# Lab 0 - Task 0.2 - Transfer Learning

### Task 0.2.1 - Transfer Learning from ImageNet (AlexNet on CIFAR-10)
- Experiment A: Fine-tuning AlexNet (train all weights)
- Experiment B: Feature Extraction (pretrained weights frozen, only new FC layer trains)

### Task 0.2.2 - Transfer Learning from MNIST to SVHN
- Train CNN on MNIST, then use as pretrained model for SVHN

### Tracking: Weights & Biases (WandB)

---
## Setup & Imports

In [2]:
%%time

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import wandb
import numpy as np

# Limit GPU memory to 25% 
torch.cuda.set_per_process_memory_fraction(0.25, device=0)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda
CPU times: user 240 μs, sys: 14 μs, total: 254 μs
Wall time: 268 μs


---
## Task 0.2.1 — AlexNet on CIFAR-10

### Load CIFAR-10

AlexNet expects input images of at least 63x63. CIFAR-10 is 32x32, so we resize to 224x224 (standard ImageNet size).

In [3]:
%%time

# AlexNet was trained on ImageNet with 224x224 images
# CIFAR-10 is 32x32 so we resize + use ImageNet normalization stats
cifar_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

cifar_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=cifar_transform
)
cifar_testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=cifar_transform
)

cifar_trainloader = torch.utils.data.DataLoader(
    cifar_trainset, batch_size=64, shuffle=True, num_workers=2
)
cifar_testloader = torch.utils.data.DataLoader(
    cifar_testset, batch_size=64, shuffle=False, num_workers=2
)

print('CIFAR-10 Training samples:', len(cifar_trainset))
print('CIFAR-10 Test samples:', len(cifar_testset))

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


CIFAR-10 Training samples: 50000
CIFAR-10 Test samples: 10000
CPU times: user 1.51 s, sys: 326 ms, total: 1.84 s
Wall time: 1.83 s


### Shared Training & Evaluation Functions 

In [4]:
%%time

NUM_EPOCHS = 10

def train_model(model, optimizer, trainloader, num_epochs, run_name, project_name="Lab-0", config=None):
    """Training function with WandB logging."""
    # --- WandB: Initialize a new run ---
    wandb.init(
        project=project_name,
        name=run_name,
        config=config or {}
    )

    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total

        # --- WandB: Log training metrics each epoch ---
        wandb.log({
            "training_loss": epoch_loss,
            "training_accuracy": epoch_acc,
            "learning_rate": optimizer.param_groups[0]['lr'],
            "epoch": epoch + 1
        })

        print(f'Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')

    # --- WandB: Close the run ---
    wandb.finish()
    print('Training complete!')
    return model


def evaluate_model(model, testloader, label):
    """Evaluation function."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f'Test Accuracy ({label}): {acc:.2f}%')
    return acc


# Dictionary to collect all results
results = {}

CPU times: user 16 μs, sys: 0 ns, total: 16 μs
Wall time: 23.6 μs


### Experiment A — Fine-Tuning AlexNet
Load AlexNet **without** pretrained weights, replace the final classifier layer with one that outputs 10 classes, and train all weights from scratch on CIFAR-10.

In [5]:
%%time

# Load AlexNet WITHOUT pretrained weights (fine-tuning = train from scratch on CIFAR-10)
alexnet_ft = models.alexnet(weights=None)

# Replace the last FC layer: AlexNet's classifier[6] outputs 1000 (ImageNet classes)
# We need 10 outputs for CIFAR-10
alexnet_ft.classifier[6] = nn.Linear(4096, 10)
alexnet_ft = alexnet_ft.to(device)

optimizer_ft = optim.Adam(alexnet_ft.parameters(), lr=0.0001)

print('=== Experiment A: AlexNet Fine-Tuning (no pretrained weights) ===')
alexnet_ft = train_model(
    model=alexnet_ft,
    optimizer=optimizer_ft,
    trainloader=cifar_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='alexnet_finetuning',
    config={
        "architecture": "AlexNet",
        "mode": "fine_tuning",
        "dataset": "CIFAR-10",
        "pretrained": False,
        "optimizer": "Adam",
        "learning_rate": 0.0001,
        "epochs": NUM_EPOCHS
    }
)

=== Experiment A: AlexNet Fine-Tuning (no pretrained weights) ===


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhashr-3 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch [1/10] | Loss: 1.5011 | Train Acc: 44.62%
Epoch [2/10] | Loss: 1.0011 | Train Acc: 64.30%
Epoch [3/10] | Loss: 0.7762 | Train Acc: 72.56%
Epoch [4/10] | Loss: 0.6322 | Train Acc: 77.92%
Epoch [5/10] | Loss: 0.5419 | Train Acc: 81.13%
Epoch [6/10] | Loss: 0.4568 | Train Acc: 84.00%
Epoch [7/10] | Loss: 0.3917 | Train Acc: 86.20%
Epoch [8/10] | Loss: 0.3229 | Train Acc: 88.67%
Epoch [9/10] | Loss: 0.2695 | Train Acc: 90.57%
Epoch [10/10] | Loss: 0.2218 | Train Acc: 92.08%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
training_accuracy,▁▄▅▆▆▇▇▇██
training_loss,█▅▄▃▃▂▂▂▁▁
epoch,10
learning_rate,0.0001
training_accuracy,92.078
training_loss,0.22181


Training complete!
CPU times: user 5min 17s, sys: 1min 32s, total: 6min 50s
Wall time: 21min 30s


In [6]:
%%time

results['Experiment A (AlexNet Fine-Tuning)'] = evaluate_model(
    alexnet_ft, cifar_testloader, 'AlexNet Fine-Tuning'
)

Test Accuracy (AlexNet Fine-Tuning): 82.51%
CPU times: user 3.01 s, sys: 2.1 s, total: 5.11 s
Wall time: 10.2 s


### Experiment B — AlexNet as Feature Extractor (Pretrained, Optional)
Load AlexNet **with** pretrained ImageNet weights. Freeze all layers except the new final classifier layer.
This way the pretrained features are reused and only the 10-class output layer is trained.

In [7]:
%%time

# Load AlexNet WITH pretrained ImageNet weights
alexnet_fe = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Freeze ALL parameters (we don't want to update pretrained features)
for param in alexnet_fe.parameters():
    param.requires_grad = False

# Replace + unfreeze only the final layer (10 classes for CIFAR-10)
alexnet_fe.classifier[6] = nn.Linear(4096, 10)  # This new layer has requires_grad=True by default
alexnet_fe = alexnet_fe.to(device)

# Only pass the trainable parameters (the new final layer) to the optimizer
optimizer_fe = optim.Adam(
    filter(lambda p: p.requires_grad, alexnet_fe.parameters()),
    lr=0.0001
)

print('=== Experiment B: AlexNet Feature Extraction (pretrained ImageNet weights) ===')
alexnet_fe = train_model(
    model=alexnet_fe,
    optimizer=optimizer_fe,
    trainloader=cifar_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='alexnet_feature_extraction',
    config={
        "architecture": "AlexNet",
        "mode": "feature_extraction",
        "dataset": "CIFAR-10",
        "pretrained": True,
        "frozen_layers": "all except classifier[6]",
        "optimizer": "Adam",
        "learning_rate": 0.0001,
        "epochs": NUM_EPOCHS
    }
)

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100.0%


=== Experiment B: AlexNet Feature Extraction (pretrained ImageNet weights) ===


Epoch [1/10] | Loss: 0.9235 | Train Acc: 69.53%
Epoch [2/10] | Loss: 0.6690 | Train Acc: 76.89%
Epoch [3/10] | Loss: 0.6207 | Train Acc: 78.38%
Epoch [4/10] | Loss: 0.5938 | Train Acc: 79.32%
Epoch [5/10] | Loss: 0.5726 | Train Acc: 80.16%
Epoch [6/10] | Loss: 0.5609 | Train Acc: 80.38%
Epoch [7/10] | Loss: 0.5484 | Train Acc: 80.85%
Epoch [8/10] | Loss: 0.5407 | Train Acc: 81.00%
Epoch [9/10] | Loss: 0.5301 | Train Acc: 81.44%
Epoch [10/10] | Loss: 0.5269 | Train Acc: 81.44%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
training_accuracy,▁▅▆▇▇▇████
training_loss,█▄▃▂▂▂▁▁▁▁
epoch,10
learning_rate,0.0001
training_accuracy,81.442
training_loss,0.52693


Training complete!
CPU times: user 2min 46s, sys: 1min 48s, total: 4min 35s
Wall time: 8min 24s


In [8]:
%%time

results['Experiment B (AlexNet Feature Extraction)'] = evaluate_model(
    alexnet_fe, cifar_testloader, 'AlexNet Feature Extraction (Pretrained)'
)

Test Accuracy (AlexNet Feature Extraction (Pretrained)): 82.88%
CPU times: user 3.08 s, sys: 2.13 s, total: 5.21 s
Wall time: 10.5 s


### Why is there a difference between Experiment A and B?

**Experiment A (Fine-Tuning / no pretrained weights):** The model starts with random weights and must learn all features from scratch on CIFAR-10. With only 10 epochs, the model may underfit since AlexNet is large and needs more data/time to converge from random initialization.

**Experiment B (Feature Extraction / pretrained weights):** The convolutional layers already contain rich feature detectors learned from 1.2 million ImageNet images (edges, textures, shapes, object parts). Even though CIFAR-10 images are different from ImageNet, these low-level features transfer well. Only the final classification layer is trained, so the model converges much faster and often achieves better accuracy with fewer epochs.

In short: **pretrained weights give the model a head start** by providing good feature representations before training even begins.

---
## Task 0.2.2 — Transfer Learning: MNIST → SVHN

### Load MNIST and SVHN datasets

In [10]:
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 78.3 MB/s  0:00:00 eta 0:00:01


In [11]:
%%time

# MNIST: grayscale 28x28 images, 10 digit classes
# We resize to 32x32 and convert to 3 channels so the same CNN works on both datasets
mnist_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.Grayscale(num_output_channels=3),  # Convert 1-channel to 3-channel
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# SVHN: color 32x32 images of house number digits, 10 classes
svhn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# MNIST
mnist_trainset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=mnist_transform
)
mnist_testset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=mnist_transform
)
mnist_trainloader = torch.utils.data.DataLoader(
    mnist_trainset, batch_size=64, shuffle=True, num_workers=2
)
mnist_testloader = torch.utils.data.DataLoader(
    mnist_testset, batch_size=64, shuffle=False, num_workers=2
)

# SVHN
svhn_trainset = torchvision.datasets.SVHN(
    root='./data', split='train', download=True, transform=svhn_transform
)
svhn_testset = torchvision.datasets.SVHN(
    root='./data', split='test', download=True, transform=svhn_transform
)
svhn_trainloader = torch.utils.data.DataLoader(
    svhn_trainset, batch_size=64, shuffle=True, num_workers=2
)
svhn_testloader = torch.utils.data.DataLoader(
    svhn_testset, batch_size=64, shuffle=False, num_workers=2
)

print('MNIST Training samples:', len(mnist_trainset))
print('MNIST Test samples:', len(mnist_testset))
print('SVHN Training samples:', len(svhn_trainset))
print('SVHN Test samples:', len(svhn_testset))

100.0%


MNIST Training samples: 60000
MNIST Test samples: 10000
SVHN Training samples: 73257
SVHN Test samples: 26032
CPU times: user 5.17 s, sys: 1.15 s, total: 6.32 s
Wall time: 11.5 s


### Define CNN (same architecture as labmate's Task 0.1)

In [14]:
%%time

class CNN(nn.Module):
    def __init__(self, activation_fn=nn.LeakyReLU()):
        super(CNN, self).__init__()
        self.activation_fn = activation_fn

        # Convolutional layers
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2)
        )

        # Fully connected layers
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            self.activation_fn,
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

CPU times: user 282 μs, sys: 105 μs, total: 387 μs
Wall time: 398 μs


### Step 1 — Train CNN on MNIST

In [13]:
%%time

cnn_mnist = CNN(activation_fn=nn.LeakyReLU()).to(device)
optimizer_mnist = optim.Adam(cnn_mnist.parameters(), lr=0.001)

print('=== Training CNN on MNIST ===')
cnn_mnist = train_model(
    model=cnn_mnist,
    optimizer=optimizer_mnist,
    trainloader=mnist_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='cnn_mnist',
    config={
        "architecture": "CNN",
        "dataset": "MNIST",
        "activation": "LeakyReLU",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "epochs": NUM_EPOCHS
    }
)

=== Training CNN on MNIST ===


Epoch [1/10] | Loss: 0.1391 | Train Acc: 95.61%
Epoch [2/10] | Loss: 0.0394 | Train Acc: 98.79%
Epoch [3/10] | Loss: 0.0277 | Train Acc: 99.16%
Epoch [4/10] | Loss: 0.0197 | Train Acc: 99.35%
Epoch [5/10] | Loss: 0.0173 | Train Acc: 99.44%
Epoch [6/10] | Loss: 0.0139 | Train Acc: 99.57%
Epoch [7/10] | Loss: 0.0126 | Train Acc: 99.60%
Epoch [8/10] | Loss: 0.0101 | Train Acc: 99.69%
Epoch [9/10] | Loss: 0.0096 | Train Acc: 99.68%
Epoch [10/10] | Loss: 0.0099 | Train Acc: 99.71%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
training_accuracy,▁▆▇▇██████
training_loss,█▃▂▂▁▁▁▁▁▁
epoch,10
learning_rate,0.001
training_accuracy,99.71333
training_loss,0.00994


Training complete!
CPU times: user 1min 7s, sys: 8.52 s, total: 1min 16s
Wall time: 2min 6s


In [15]:
%%time

results['MNIST CNN'] = evaluate_model(cnn_mnist, mnist_testloader, 'CNN on MNIST')

Test Accuracy (CNN on MNIST): 99.34%
CPU times: user 412 ms, sys: 275 ms, total: 687 ms
Wall time: 2.25 s


### Step 2 — Transfer the MNIST CNN to SVHN (Feature Extraction)

Freeze the convolutional layers (they've already learned useful features), keep the FC layers trainable, and fine-tune on SVHN.

In [16]:
%%time

# Start from the MNIST-trained CNN
cnn_svhn = CNN(activation_fn=nn.LeakyReLU()).to(device)

# Copy the MNIST weights into the SVHN model
cnn_svhn.load_state_dict(cnn_mnist.state_dict())

# Freeze the conv_block (pretrained on MNIST)
for param in cnn_svhn.conv_block.parameters():
    param.requires_grad = False

# Only train the fc_block on SVHN
optimizer_svhn = optim.Adam(
    filter(lambda p: p.requires_grad, cnn_svhn.parameters()),
    lr=0.001
)

print('=== Transfer Learning: MNIST CNN → SVHN ===')
cnn_svhn = train_model(
    model=cnn_svhn,
    optimizer=optimizer_svhn,
    trainloader=svhn_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='cnn_mnist_to_svhn',
    config={
        "architecture": "CNN",
        "mode": "transfer_learning",
        "source_dataset": "MNIST",
        "target_dataset": "SVHN",
        "frozen_layers": "conv_block",
        "trainable_layers": "fc_block",
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "epochs": NUM_EPOCHS
    }
)

=== Transfer Learning: MNIST CNN → SVHN ===


Epoch [1/10] | Loss: 1.1415 | Train Acc: 63.73%
Epoch [2/10] | Loss: 0.8271 | Train Acc: 73.61%
Epoch [3/10] | Loss: 0.7161 | Train Acc: 77.10%
Epoch [4/10] | Loss: 0.6382 | Train Acc: 79.63%
Epoch [5/10] | Loss: 0.5782 | Train Acc: 81.46%
Epoch [6/10] | Loss: 0.5284 | Train Acc: 83.30%
Epoch [7/10] | Loss: 0.4876 | Train Acc: 84.43%
Epoch [8/10] | Loss: 0.4506 | Train Acc: 85.75%
Epoch [9/10] | Loss: 0.4193 | Train Acc: 86.72%
Epoch [10/10] | Loss: 0.3918 | Train Acc: 87.64%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
training_accuracy,▁▄▅▆▆▇▇▇██
training_loss,█▅▄▃▃▂▂▂▁▁
epoch,10
learning_rate,0.001
training_accuracy,87.63941
training_loss,0.39182


Training complete!
CPU times: user 55.1 s, sys: 8.9 s, total: 1min 4s
Wall time: 1min 41s


In [17]:
%%time

results['SVHN (Transfer from MNIST)'] = evaluate_model(cnn_svhn, svhn_testloader, 'CNN Transfer MNIST→SVHN')

Test Accuracy (CNN Transfer MNIST→SVHN): 77.79%
CPU times: user 1.27 s, sys: 430 ms, total: 1.7 s
Wall time: 3.84 s


---
## Final Results Summary

In [20]:
print(f"{'Experiment':<45} {'Test Accuracy':>15}")
print('-' * 62)

# Loop with matching alignment
for name, acc in results.items():
    print(f"{name:<45} {acc:>14.2f}%")

Experiment                                      Test Accuracy
--------------------------------------------------------------
Experiment A (AlexNet Fine-Tuning)                     82.51%
Experiment B (AlexNet Feature Extraction)              82.88%
MNIST CNN                                              99.34%
SVHN (Transfer from MNIST)                             77.79%
